## import

In [1]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from tensorflow.keras import layers, models
from tensorflow_model_optimization.python.core.keras.compat import keras
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import BatchNormalization, Input
from tensorflow_model_optimization.quantization.keras import quantize_annotate_layer, quantize_annotate_model
import numpy as np
from pycoral.utils.edgetpu import make_interpreter
from pycoral.adapters.common import input_size
from pycoral.adapters.classify import get_classes
import time
from pycoral.pybind._pywrap_coral import SetVerbosity as set_verbosity
from utils import log_model_info, load_configs
import os

## Load Train Data

In [2]:
## Load configurations
train_config = load_configs(r"E:\MTP_DATA\Final-MTP\config\train.yaml")
size = train_config['image']['img_size']

## Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize and reshape data
x_train = x_train / 255.0
x_test = x_test / 255.0
x_train = x_train.reshape(-1, size, size, 1)  # Add channel dimension (grayscale)
x_test = x_test.reshape(-1, size, size, 1)
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
y_train_one_hot = tf.keras.utils.to_categorical(y_train, 10)
y_test_one_hot = tf.keras.utils.to_categorical(y_test, 10)

## Build Model

In [9]:
## load model configurations
model_config = load_configs(r"E:\MTP_DATA\Final-MTP\config\model.yaml")
EPOCHS = train_config['model']['epochs']
BATCH_SIZE = train_config['model']['batch_size']
LOSS = model_config['loss']
OPTIMIZER = model_config['optimizer']
MATRICS = model_config['metrics']
MODEL_DIR = train_config['dir']['save_dir']
MODEL_NAME = model_config['name']
LOG_FILE = model_config['log']
CPU_MODEL = os.path.join(MODEL_DIR, MODEL_NAME + '.h5')  # Full path to save the model
TFLITE_MODEL = os.path.join(MODEL_DIR, MODEL_NAME + '.tflite')  # Full path to save the TFLite model
TPU_MODEL = os.path.join(MODEL_DIR, MODEL_NAME + '_edgetpu.tflite')  # Full path to save the TPU model

# def create_model():
#     model = keras.Sequential([
#         keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
#         keras.layers.Flatten(),
#         keras.layers.Dense(10, activation='softmax')
#     ])
#     return model

def create_model():
    model = keras.Sequential([
        keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        keras.layers.Conv2D(64, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(128, (3, 3), activation='relu'),
        keras.layers.Conv2D(128, (3, 3), activation='relu'),
        keras.layers.Flatten(),
        keras.layers.Dense(1024, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    return model


model = create_model()
model.compile(optimizer=OPTIMIZER,
                   loss=LOSS,
                   metrics=MATRICS)

## Training Base Line Model

In [10]:
print("Training baseline model...")
# print(x_train.shape, y_train.shape)
model.fit(x_train, y_train_one_hot, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(x_test, y_test_one_hot))
model.save(CPU_MODEL)
print("Model saved to:", CPU_MODEL)

Training baseline model...
Epoch 1/3
938/938 [==============================] - 148s 157ms/step - loss: 0.1029 - accuracy: 0.9683 - val_loss: 9.8092 - val_accuracy: 0.9806
Epoch 2/3
938/938 [==============================] - 189s 202ms/step - loss: 0.0344 - accuracy: 0.9895 - val_loss: 19.2516 - val_accuracy: 0.9661
Epoch 3/3
938/938 [==============================] - 156s 166ms/step - loss: 0.0242 - accuracy: 0.9925 - val_loss: 19.4504 - val_accuracy: 0.9760


e:\MTP_DATA\Scripts\qat-env\lib\site-packages\tf_keras\src\engine\training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model saved to: E:\MTP_DATA\Final-MTP\Models\complex_mnist_model.h5


## Quantization Aware Training

In [6]:
# model = load_model(model_path)
qat_model = tfmot.quantization.keras.quantize_model(model)
qat_model.compile(optimizer=OPTIMIZER,
                  loss=LOSS,
                  metrics=MATRICS)

# # Fine-tune the QAT model
print("Training QAT model...")
qat_model.fit(x_train, y_train_one_hot, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(x_test, y_test_one_hot))
log_model_info(MODEL_NAME, model, OPTIMIZER, str(EPOCHS), str(BATCH_SIZE), LOG_FILE)

# Evaluate on test data
test_loss, test_acc = model.evaluate(x_test, y_test_one_hot)
print(f"QAT Model Test Accuracy: {test_acc:.4f}")

# CPU_MODEL = os.path.join(MODEL_DIR + MODEL_NAME + ".h5")



Training QAT model...
Epoch 1/3
938/938 [==============================] - 182s 192ms/step - loss: 0.0211 - accuracy: 0.9934 - val_loss: 0.0343 - val_accuracy: 0.9903
Epoch 2/3
938/938 [==============================] - 342s 365ms/step - loss: 0.0144 - accuracy: 0.9958 - val_loss: 0.0234 - val_accuracy: 0.9925
Epoch 3/3
938/938 [==============================] - 197s 209ms/step - loss: 0.0112 - accuracy: 0.9963 - val_loss: 0.0277 - val_accuracy: 0.9915
✅ Appended model info for 'complex_mnist_model' to E:\\MTP_DATA\\Final-MTP\\log\\mnist.txt
313/313 [==============================] - 8s 24ms/step - loss: 0.0362 - accuracy: 0.9890
QAT Model Test Accuracy: 0.9890


## Model Conversion

In [11]:
# Define a representative dataset function
(x_train, _), _ = tf.keras.datasets.mnist.load_data()

# Normalize and reshape to match model input
x_train = x_train / 255.0
x_train = x_train.reshape(-1, 28, 28, 1).astype(np.float32)  # Must be float32 initially


def representative_data_gen():
    for i in range(100):  # Use a small subset (100 samples)
        yield [x_train[i:i+1]]  # Return as a list

# Load the trained QAT model
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)

# ✅ Enable full INT8 quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen  # Needed for proper calibration

# ✅ Ensure both inputs and outputs are INT8
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8  # Force input to UINT8
converter.inference_output_type = tf.uint8 # Keep output as float32

# Convert and save the INT8 model
quantized_int8_tflite_model = converter.convert()
with open(TFLITE_MODEL, "wb") as f:
    f.write(quantized_int8_tflite_model)

print("Full INT8 quantized model saved as {}".format(TFLITE_MODEL))


INFO:tensorflow:Assets written to: C:\Users\DELL\AppData\Local\Temp\tmppl9j2tyk\assets


INFO:tensorflow:Assets written to: C:\Users\DELL\AppData\Local\Temp\tmppl9j2tyk\assets
e:\MTP_DATA\Scripts\qat-env\lib\site-packages\tensorflow\lite\python\convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Full INT8 quantized model saved as E:\MTP_DATA\Final-MTP\Models\complex_mnist_model.tflite


## CPU + TPU Inference

In [12]:
import tensorflow as tf
import numpy as np
import time
import csv
import os
from utils import load_configs

# ---------- Configuration ----------
test_config = load_configs(r"E:\MTP_DATA\Final-MTP\config\test.yaml")
MODEL_DIR = test_config['model']['model_dir']
MODEL_NAME = test_config['model']['model_name']
DEVICE = test_config['device']
if DEVICE == 'cpu' or DEVICE == 'tflite':
    DEVICE = 'tflite'
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME + test_config['extantion'][DEVICE])
IMG_SIZE = test_config['image']['img_size']
RESULT_DIR = test_config['results']['save_dir']  # Directory to save the results
if not os.path.exists(RESULT_DIR):  # Create the directory if it doesn't exist
    os.makedirs(RESULT_DIR, exist_ok=True)


def resize_dataset(x, target_size):
    # Resize the images to the target size while ensuring they retain their original number of channels (1 for grayscale).
    x_resized = tf.image.resize(x, [target_size, target_size])  # Resize to the target size
    x_resized = tf.clip_by_value(x_resized, 0.0, 1.0)  # Clip values to the valid range [0, 1]
    
    # Convert to uint8 format
    return np.uint8(x_resized.numpy() * 255)  # Convert to [0, 255] and cast to uint8

# ---------- Load test dataset ----------
(_, _), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_test = x_test.reshape(-1, 28, 28, 1).astype('uint8')  # Expected input dtype

x_test = resize_dataset(x_test, IMG_SIZE)  # Resize the dataset to the target size
# ---------- Load TFLite model ----------
interpreter = None
if DEVICE == 'tflite':
    interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
else:
    interpreter = make_interpreter(MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# ---------- Helper to count total model parameters ----------
def count_tflite_parameters(interpreter):
    total_params = 0
    for detail in interpreter.get_tensor_details():
        shape = detail['shape']
        if shape is not None and len(shape) > 0:
            num_params = np.prod(shape)
            total_params += num_params
    return total_params

total_params = count_tflite_parameters(interpreter)

# ---------- Inference function ----------
def predict(image):
    interpreter.set_tensor(input_details[0]['index'], np.expand_dims(image, axis=0))
    start = time.time()
    interpreter.invoke()
    end = time.time()
    output = interpreter.get_tensor(output_details[0]['index'])
    return np.argmax(output), end - start

# ---------- Run inference on test set ----------
correct_predictions = 0
total_time = 0
for i in range(len(x_test)):
    if i % 1000 == 0:
        print(f"Processing image {i}...")
    pred_label, time_taken = predict(x_test[i])
    total_time += time_taken
    if pred_label == y_test[i]:
        correct_predictions += 1

accuracy = correct_predictions / len(x_test) * 100
avg_time_per_image = total_time / len(x_test) * 1000  # in ms

print(f"TFLite Model Accuracy: {accuracy:.2f}%")
print(f"Total Time Taken: {total_time:.4f} s")
print(f"Avg Time per Image: {avg_time_per_image:.2f} ms")
print(f"Average Frames per Second (FPS): {len(x_test) / total_time:.2f} FPS")

# ---------- Save results to CSV ----------
# header = ["Model_Name", "Model_Size", "Total_Parameters", "Total_Time(s)", "Avg_Time_per_Image(ms)", "Accuracy(%)", "Device"]
# row = [model_name, f"{model_size:.2f}", total_params, f"{total_time:.4f}", f"{avg_time_per_image:.2f}", f"{accuracy:.2f}", device]

header = ["Model_Name","Input_Size", "Total_Parameters", "Total_Time(s)", "Avg_Time_per_Image(ms)", "Accuracy(%)", "FPS", "Device"]
row = [MODEL_NAME, IMG_SIZE, total_params, f"{total_time:.4f}", f"{avg_time_per_image:.2f}", f"{accuracy:.2f}", f"{len(x_test) / total_time:.2f}",DEVICE]
results_file = os.path.join(RESULT_DIR,"mnist_results.csv")
write_header = not os.path.exists(results_file)
with open(results_file, mode="a", newline='') as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(header) 
    writer.writerow(row)

print(f"\n✅ Results saved to '{results_file}'")


e:\MTP_DATA\Scripts\qat-env\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Processing image 0...
Processing image 1000...
Processing image 2000...
Processing image 3000...
Processing image 4000...
Processing image 5000...
Processing image 6000...
Processing image 7000...
Processing image 8000...
Processing image 9000...
TFLite Model Accuracy: 98.26%
Total Time Taken: 7.1349 s
Avg Time per Image: 0.71 ms
Average Frames per Second (FPS): 1401.56 FPS

✅ Results saved to 'E:\MTP_DATA\Final-MTP\Results\mnist_results.csv'
